# Walmart : predict weekly sales

## Company's Description

Walmart Inc. is an American multinational retail corporation that operates a chain of hypermarkets, discount department stores, and grocery stores from the United States, headquartered in Bentonville, Arkansas. The company was founded by Sam Walton in 1962.

## Project

Walmart's marketing service would like to build a machine learning model able to estimate the weekly sales in their stores, with the best precision possible on the predictions made. Such a model would help them understand better how the sales are influenced by economic indicators, and might be used to plan future marketing campaigns.

## Goals

The project can be divided into the following steps:

- **Initial exploration and basic statistics** of the dataset
- **Preprocessings with Pandas and Scikit-learn** to prepare data for machine learning
- **Exploratory data analysis**
- **Model training**:
    - Baseline model: **linear regression model**
    - Training **regularized regression models (Ridge, Lasso)** to avoid overfitting

## Dataset

The dataset used for this project contains information about weekly sales achieved by different Walmart stores, and other variables such as the unemployment rate or the fuel price, that might be useful for predicting the amount of sales.


In [1]:
# Import useful packages and modules
import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Import dataset
df_raw = pd.read_csv('data/Walmart_Store_sales.csv')

# 1. Overview of the raw dataset

In [3]:
print("Display of the first lines of the dataset: ")
display(df_raw.head())

Display of the first lines of the dataset: 


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [4]:
print(f"Shape of the dataframe (number of rows, number of columns) : {format(df_raw.shape)}\n")

print("Display of the types of the columns :")
print(df_raw.info())

Shape of the dataframe (number of rows, number of columns) : (150, 8)

Display of the types of the columns :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB
None


In [5]:
print("Basic statistics: ")
display(df_raw.describe(include = "all"))
print()

Basic statistics: 


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


In [6]:
# Missing values
df_count_na = df_raw.isnull().sum().reset_index().rename(columns = {'index' : 'var_name', 0: "count_of_missing_values"})
df_count_na['proportion_of_missing_values'] = round(df_count_na['count_of_missing_values'] / len(df_raw) * 100, 2)
df_count_na.sort_values(['proportion_of_missing_values'], 
                        ascending = False,
                        inplace = True)
df_count_na

,var_name,count_of_missing_values,proportion_of_missing_values
1,Date,18,12.00
4,Temperature,18,12.00
7,Unemployment,15,10.00
2,Weekly_Sales,14,9.33
5,Fuel_Price,14,9.33
3,Holiday_Flag,12,8.00
6,CPI,12,8.00
0,Store,0,0.00


In [7]:
# Duplicates
# We check for duplicates in Store x Date, in the dataset where missing values in store or date are excluded.

nb_dup = df_raw.dropna(subset=['Store','Date']).duplicated(['Store', 'Date']).sum()

print(f"The dataframe contains {nb_dup} duplicates of store x date.")

The dataframe contains 0 duplicates of store x date.


In [8]:
# Values taken by Store variable
df_raw["Store"].value_counts()

Store
3.0     15
1.0     11
18.0    10
13.0     9
5.0      9
19.0     9
14.0     9
17.0     8
2.0      8
8.0      8
7.0      8
6.0      7
20.0     7
4.0      7
10.0     5
12.0     5
16.0     4
15.0     4
9.0      4
11.0     3
Name: count, dtype: int64

In [9]:
# Convert Date column from string to datetime
df_raw['Date'] = pd.to_datetime(df_raw['Date'],
                            format = '%d-%m-%Y')
# Period of analysis
print("Period : from ", df_raw['Date'].min(), "to ",  df_raw['Date'].max())

Period : from  2010-02-05 00:00:00 to  2012-10-19 00:00:00


We are dealing with **panel data**: data is collected for 20 stores over a period of 2 years and 8 months, from 05/02/2010 to 19/10/2012.

The dataset has a limited number of observations (150). 

The panel dataset is **imbalanced**: all stores are not observed over the same period of time. Indeed, the number of observations per store is inequal: store 11 has 3 observations whereas store 3 has 15 observations.

There are **no duplicates** in the dataframe: each pair (store, observation date) is unique. 

Except for the store id, there are **missing values** in every other column, between 8% and 12%, which is not an excessive rate. We will deal with them later during the preprocessing stage. 

## 2. Preprocessing with Pandas

The pre-processing steps will be the following:

**Drop lines where target values are missing :**
 - The target variable (Y) corresponds to the column *Weekly_Sales*. One can see above that there are some missing values in this column.
 - We never use imputation techniques on the target : it might create some bias in the predictions
 - Then, we will just drop the lines in the dataset for which the value in *Weekly_Sales* is missing.
 
**Create usable features from the *Date* column :**
The *Date* column cannot be included as it is in the model. One solution is to drop this column. Another one is to create new columns that contain the following numeric features : *year*, *month*, *day*, *week of year*, *day of week*.

**Handling missing values of Holiday flag**: imputation of the value 0 after analysis of the associated dates

**Drop lines containing invalid values or outliers :**
In this project, will be considered as outliers all the numeric features that don't fall within the range : $[\bar{X} - 3\sigma, \bar{X} + 3\sigma]$. The following variables are concerned: *Temperature*, *Fuel_price*, *CPI* and *Unemployment*


In [10]:
df = df_raw.copy()

In [11]:
# Define the target variable
target = "Weekly_Sales"

In [12]:
# Reformat the store variable, from integer to string
print("Before formatting, the store variable takes the following values : ", df['Store'].unique())
df['Store'] = df['Store'].astype(int).astype(str)
print("After formatting, the store variable takes the following values : ", df['Store'].unique())

Before formatting, the store variable takes the following values :  [ 6. 13. 17. 11.  4. 15. 20. 14.  3.  8. 18.  7.  1.  2.  5. 19. 16. 12.
  9. 10.]
After formatting, the store variable takes the following values :  ['6' '13' '17' '11' '4' '15' '20' '14' '3' '8' '18' '7' '1' '2' '5' '19'
 '16' '12' '9' '10']


In [13]:
# Scale down the weekly sales: convert from USD to USD millions 
# It will help readability and reduce the range of values with the other values of the dataframe
df['Weekly_Sales'] = df['Weekly_Sales'] / 10**6

In [14]:
# Drop lines where target values are missing 
df = df.loc[~df[target].isnull()]
print(f"Number of remaining rows of the dataframe after removing missing values of target: {len(df)}")

# Drop lines where Date are missing 
df = df.loc[~df['Date'].isnull()]
print(f"Number of remaining rows of the dataframe after removing missing values of Date: {len(df)}")

Number of remaining rows of the dataframe after removing missing values of target: 136
Number of remaining rows of the dataframe after removing missing values of Date: 118


In [15]:
# Create features from the Date column: year, month, day, day of week, week of the year
df['Year'] = df['Date'].dt.year.astype(int)
df['Month'] = df['Date'].dt.month.astype(int)
df['Day'] = df['Date'].dt.day.astype(int)
df['DayOfWeek'] = df['Date'].dt.weekday # 0 for Monday, 6 for Sunday
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek,WeekOfYear
0,6,2011-02-18,1.572118,NaN,59.61,3.045,214.777523,6.858,2011,2,18,4,7
1,13,2011-03-25,1.807545,0.0,42.38,3.435,128.616064,7.470,2011,3,25,4,12
4,6,2010-05-28,1.644471,0.0,78.89,2.759,212.412888,7.092,2010,5,28,4,21
5,4,2010-05-28,1.857534,0.0,NaN,2.756,126.160226,7.896,2010,5,28,4,21
6,15,2011-06-03,0.695396,0.0,69.80,4.069,134.855161,7.658,2011,6,3,4,22


### Imputation of missing *Holiday_Flag* values

In [16]:
# Displaying observations occuring during holidays
df.loc[df['Holiday_Flag'] == 1].sort_values('Date')

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek,WeekOfYear
44,1,2010-02-12,1.641957,1.0,38.51,2.548,211.242170,8.106,2010,2,12,4,6
107,8,2010-02-12,0.994801,1.0,33.34,2.548,214.621419,6.299,2010,2,12,4,6
135,12,2010-09-10,0.903119,1.0,83.63,3.044,126.114581,14.180,2010,9,10,4,36
114,11,2010-11-26,1.757243,1.0,69.90,2.735,215.061403,7.564,2010,11,26,4,47
110,20,2010-12-31,1.799738,1.0,28.85,3.179,204.643227,7.484,2010,12,31,4,52
32,7,2012-02-10,0.563461,1.0,18.79,3.103,196.919506,8.256,2012,2,10,4,6
33,14,2012-02-10,2.077256,1.0,37.00,NaN,NaN,8.424,2012,2,10,4,6
122,7,2012-09-07,0.597877,1.0,57.84,3.596,198.095048,7.872,2012,9,7,4,36


In [17]:
fig = px.scatter(
    df, 
    x="Date", 
    y="Holiday_Flag", 
    title="Holidays", 
    height=350
)

fig.update_layout(
    title={
        'x': 0.5,
    }
)

fig.show()

In [18]:
# Check whether Holiday_Flag is defined at a weekly level

# Check consistency between (Year, WeekOfYear) and Holiday_Flag in the subset with non-missing values 
countholidayflag_by_weekofyear = (df
    .dropna(subset=["Holiday_Flag", "Year", "WeekOfYear"])
    .groupby(['Year', 'WeekOfYear'])["Holiday_Flag"]
    .nunique()
)

problematic_cases = countholidayflag_by_weekofyear[countholidayflag_by_weekofyear>1]

print(f"There are {len(problematic_cases)} inconsistencies between (Year, WeekOfYear) and Holiday_Flag")

There are 0 inconsistencies between (Year, WeekOfYear) and Holiday_Flag


*Holiday_Flag* is defined at the weekly level, which means it is an approximation: if any day in a week is a holiday, the whole week is considered a holiday (Holiday_Flag = 1).
 
The dataset contains few observations occuring during holidays (8 out of 118, accounting for 7% of the database).  
We note that the periods of holidays are the following: Superbowl in February, Labour Day in September, Thanksgiving in November, Christmas in December.

In [19]:
print("Number of missing values of Holiday_Flag :", df['Holiday_Flag'].isnull().sum())
df.loc[df['Holiday_Flag'].isnull()].sort_values('Date')

Number of missing values of Holiday_Flag : 9


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek,WeekOfYear
15,6,2010-04-30,1.498080,NaN,68.91,2.780,211.894272,7.092,2010,4,30,4,17
118,9,2010-06-18,0.513074,NaN,82.99,2.637,215.016648,6.384,2010,6,18,4,24
90,9,2010-07-09,0.485389,NaN,78.51,2.642,214.656430,6.442,2010,7,9,4,27
73,1,2010-08-27,1.449143,NaN,85.22,2.619,211.567306,7.787,2010,8,27,4,34
0,6,2011-02-18,1.572118,NaN,59.61,3.045,214.777523,6.858,2011,2,18,4,7
53,14,2011-03-25,1.879451,NaN,41.76,3.625,184.994368,8.549,2011,3,25,4,12
136,4,2011-07-08,2.066542,NaN,84.59,3.469,129.112500,5.644,2011,7,8,4,27
48,1,2011-08-05,1.624384,NaN,91.65,3.684,215.544618,7.962,2011,8,5,4,31
43,7,2011-08-26,0.629994,NaN,57.60,3.485,194.379637,8.622,2011,8,26,4,34


In [20]:
# Create mapping between (Year, WeekOfYear) and Holiday_Flag

mapping_holiday = (
    df
    .dropna(subset=["Holiday_Flag", "Year", "WeekOfYear"])
    .drop_duplicates(subset=["Year", "WeekOfYear"])
    .loc[:, ["Year", "WeekOfYear", "Holiday_Flag"]]
)

mapping_holiday

,Year,WeekOfYear,Holiday_Flag
1,2011,12,0.0
4,2010,21,0.0
6,2011,22,0.0
7,2012,5,0.0
8,2010,49,0.0
...,...,...,...
138,2011,16,0.0
139,2012,21,0.0
142,2011,40,0.0
143,2010,22,0.0


In [21]:
# Join dataset with mapping
df = df.merge(
    mapping_holiday.rename(columns={"Holiday_Flag": "Holiday_Flag_imputed"}),
    on=["Year", "WeekOfYear"],
    how="left"
)

# Impute the missing values of Holiday_Flag
df["Holiday_Flag"] = df["Holiday_Flag"].fillna(df["Holiday_Flag_imputed"])
df = df.drop(columns="Holiday_Flag_imputed")


In [22]:
# Check remaining missing values of Holiday_Flag
print("Number of missing values of Holiday_Flag :", df['Holiday_Flag'].isnull().sum())
df.loc[df['Holiday_Flag'].isnull()].sort_values('Date')

Number of missing values of Holiday_Flag : 3


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek,WeekOfYear
0,6,2011-02-18,1.572118,NaN,59.61,3.045,214.777523,6.858,2011,2,18,4,7
107,4,2011-07-08,2.066542,NaN,84.59,3.469,129.112500,5.644,2011,7,8,4,27
40,1,2011-08-05,1.624384,NaN,91.65,3.684,215.544618,7.962,2011,8,5,4,31


There are 3 remaining dates with missing values.
Using https://www.timeanddate.com/calendar/, 18/02/2011 and 05/08/2011 do not fall in a week with any public holidays, so we will impute Holiday_Flag = 0 for these dates.
08/07/2011 falls during a week with one holiday (Independence Day on 04/07/2011) so we will impute Holiday_Flag = 1.

In [23]:
# Imputation of missing values of holiday_flag
df.loc[df['Date'].isin([datetime(2011, 2, 18), datetime(2011, 8, 5)]), 'Holiday_Flag'] = 0
df.loc[df['Date']==datetime(2011, 7, 8), 'Holiday_Flag'] = 1

In [24]:
# Formatting Holiday_Flag
df['Holiday_Flag'] = df['Holiday_Flag'].astype(int)
print(f"Values taken by Holiday_Flag : {df['Holiday_Flag'].unique()}")

Values taken by Holiday_Flag : [0 1]


### Identification and removal of outliers

We consider outliers as all the numeric features that don't fall within the range :
$[\bar{X} - 3\sigma, \bar{X} + 3\sigma]$. 
This potentially concerns 4 variables : *Temperature* (expressed in Fahrenheit), *Fuel_price*, *CPI* and *Unemployment*.

In [25]:
for var in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
    fig = px.histogram(
        df, 
        x=var, 
        marginal="box", 
        title=f"Distribution of {var}", 
        height=400, 
        width=700)
    fig.update_layout(title={'x': 0.5})
    fig.show()

Based on the visualisations, it seems that only the unemployment variable has outliers (observations with an unemployment rate above 13%). We will drop them in the following step.

In [26]:
# NB : we keep missing values which will be treated later during the preprocessing with scikit-learn

for var in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
    print("Checking outliers in " + var)
    to_keep = ((df[var] - df[var].mean()).abs()<= 3*df[var].std()) | (df[var].isnull())
    df = df.loc[to_keep]
    print(f"Number of outliers in {var}: {len(df.loc[~to_keep])}")
    print(f"Number of lines remaining: {len(df)}")
    print()

Checking outliers in Temperature
Number of outliers in Temperature: 0
Number of lines remaining: 118

Checking outliers in Fuel_Price
Number of outliers in Fuel_Price: 0
Number of lines remaining: 118

Checking outliers in CPI
Number of outliers in CPI: 0
Number of lines remaining: 118

Checking outliers in Unemployment
Number of outliers in Unemployment: 0
Number of lines remaining: 113



In [27]:
# Dropping useless column: DayOfWeek
# DayOfWeek is constant and always takes the value 4, meaning the weekly sales are reported on Tuesdays.
df.drop(columns=["DayOfWeek"], inplace=True)

## 3. Exploratory data analysis

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 113 entries, 0 to 117
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         113 non-null    object        
 1   Date          113 non-null    datetime64[ns]
 2   Weekly_Sales  113 non-null    float64       
 3   Holiday_Flag  113 non-null    int64         
 4   Temperature   103 non-null    float64       
 5   Fuel_Price    102 non-null    float64       
 6   CPI           104 non-null    float64       
 7   Unemployment  102 non-null    float64       
 8   Year          113 non-null    int64         
 9   Month         113 non-null    int64         
 10  Day           113 non-null    int64         
 11  WeekOfYear    113 non-null    UInt32        
dtypes: UInt32(1), datetime64[ns](1), float64(5), int64(4), object(1)
memory usage: 11.1+ KB


### Basic statistics

In [29]:
df.describe(include="all").round(2)

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,WeekOfYear
count,113,113,113.00,113.00,103.00,102.00,104.00,102.00,113.00,113.00,113.00,113.0
unique,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
top,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
freq,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
mean,NaN,2011-04-24 21:52:33.982300928,1.27,0.07,60.20,3.27,180.11,7.38,2010.83,6.27,16.53,25.16
min,NaN,2010-02-05 00:00:00,0.27,0.00,18.79,2.51,126.11,5.14,2010.00,1.00,1.00,1.0
25%,NaN,2010-07-30 00:00:00,0.56,0.00,45.02,2.81,132.58,6.64,2010.00,4.00,10.00,15.0
50%,NaN,2011-04-22 00:00:00,1.42,0.00,61.11,3.30,197.50,7.40,2011.00,6.00,17.00,25.0
75%,NaN,2012-01-13 00:00:00,1.85,0.00,75.26,3.68,214.81,8.10,2012.00,9.00,24.00,36.0
max,NaN,2012-10-19 00:00:00,2.77,1.00,91.65,4.17,226.97,9.52,2012.00,12.00,31.00,52.0


### 3.1 Univariate analysis

In [30]:
fig = px.histogram(df, 
                x="Weekly_Sales", 
                marginal="box", 
                width=700, 
                height=400,
                title="Weekly sales distribution")
fig.update_layout(title={'x': 0.5})
fig.show()

Stores are heterogenous in terms of sales. The range of weekly sales is high: the highest sales are 10 times higher than the lowest.

In [31]:
for col in [x for x in df.columns if x!="Weekly_Sales"]:
    fig = px.histogram(
        df, 
        x=df[col], 
        width=700, 
        height=400, 
        title=f"Distribution of {col}"
        )
    fig.update_layout(title={'x': 0.5})
    fig.show()

Temperature is expressed in degrees Fahrenheit. It varies in the range from 0 to 100° Fahrenheit, corresponding to a range from -18 to 38° Celsius.  

The fuel price distribution is bimodal (two peaks of values). Prices tend to gather around two ranges: a range of low price (around 2.6-2.8) and a range of high price (3.6-3.8).

The consumer price index distribution seems also to be bimodal. Thoughout the period, the index consistently remains above 100, indicating that prices during the period are higher than prices of a reference date. 

The dataset contains more records for the year 2010 compared to 2011 and 2012 (49 records in 2010 vs 34 in 2011 and 30 in 2012). The most frequent months are June, May and February.

### 3.2 Correlation matrix

In [32]:
cols = [col for col in df.columns if col not in ("Store", "Date")]
df_corr = df[cols].corr()
display(df_corr.round(3))

,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,WeekOfYear
Weekly_Sales,1.000,0.070,-0.193,-0.022,-0.361,0.175,-0.040,-0.013,-0.036,-0.016
Holiday_Flag,0.070,1.000,-0.230,-0.141,0.107,0.024,0.015,-0.035,-0.068,-0.038
Temperature,-0.193,-0.230,1.000,-0.058,0.166,-0.205,-0.156,0.258,0.114,0.264
Fuel_Price,-0.022,-0.141,-0.058,1.000,-0.161,-0.007,0.834,-0.115,-0.045,-0.112
CPI,-0.361,0.107,0.166,-0.161,1.000,-0.164,-0.022,-0.010,0.175,0.008
Unemployment,0.175,0.024,-0.205,-0.007,-0.164,1.000,-0.174,-0.124,-0.031,-0.129
Year,-0.040,0.015,-0.156,0.834,-0.022,-0.174,1.000,-0.235,-0.134,-0.235
Month,-0.013,-0.035,0.258,-0.115,-0.010,-0.124,-0.235,1.000,-0.015,0.997
Day,-0.036,-0.068,0.114,-0.045,0.175,-0.031,-0.134,-0.015,1.000,0.067
WeekOfYear,-0.016,-0.038,0.264,-0.112,0.008,-0.129,-0.235,0.997,0.067,1.000


In [33]:
fig = go.Figure()
fig.add_trace(
    go.Heatmap(
        x=df_corr.columns,
        y=df_corr.index,
        z=np.array(df_corr),
        text=df_corr.values,
        texttemplate = '%{text:.2f}',
        colorscale="matter"
    )
)
fig.update_layout(
    title=
    {
        'text': 'Correlation matrix',
        'x': 0.5
    })
fig.show()

The correlation between *WeekOfYear* and *Month* is equal to 1: we cannot keep both variables in the modeling, we need to drop one to avoid colinearity.


Since we deal with panel data, the following observations should be interpreted with caution because they do not account for temporal effects:

* Weekly sales exhibit a slight negative correlation with the CPI and temperature, indicating that:
    - sales tend to decrease when the CPI is high: this relationship seems logical, as CPI is related to consumers' purchasing power;
    - sales tend to be lower during periods of high temperatures.  

* Weekly sales show a weak positive correlation with unemployment, which seems counterintuitive.

* *Fuel prices* and *Year* are strongly positively correlated, suggesting an increase in fuel prices over during the period.


### 3.3 Time series analysis

As the dataset has a temporal dimension, we need to look at the evolution of the variables across time.

In [34]:
# Temporal evolution of weekly sales
fig = px.scatter(df, 
                x=df['Date'], 
                y=df['Weekly_Sales'], 
                width=800, 
                height=400, 
                title=f"Weekly sales (M USD)")
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="",
    title={'x': 0.5}
    )
fig.show()

The graph of weekly sales over time is barely interpretable because of the dispersion of the observations, which may be due to variability between stores and time effects.

In [35]:
# Evolution of average weekly sales
df_avg_sales = df[['Date', 'Weekly_Sales']].groupby('Date').mean().reset_index()
fig = px.line(df_avg_sales, 
            x="Date",
            y="Weekly_Sales", 
            height=400,
            title="Average weekly sales (M USD)")
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="",
    title={'x': 0.5}
    )
fig.show()

In [36]:
## Evolution of weekly sales per store
df_sales_per_store = df[['Store','Date', 'Weekly_Sales']].groupby(['Store', 'Date']).sum().reset_index()

fig = px.line(df_sales_per_store, 
            x="Date", 
            y="Weekly_Sales", 
            color="Store",
            markers=True,
            height=600,
            title="Weekly sales per store (M USD)")
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="",
    title={'x': 0.5})
fig.show()

Weekly sales levels are heterogenous across stores. There is a clear impact of **store-specific effects** in explaining differences in weekly sales: geographic location, customer demographics, store size and range of products, store specific marketing and promotions...

At the store level, weekly sales seem to remain relatively stable over time for a majority of stores. 
A limited number of store display deviations from this pattern; we note irregularities for stores 2, 4, 13, 14 and 18. Some abrupt peaks and drops seem to be related to **seasonality**: for instance, peaks during December followed by drops in January.


In [37]:
# Boxplot of weekly sales by month (all years)
fig = px.box(df, 
                x='Month', 
                y='Weekly_Sales', 
                width=900, 
                height=500, 
                title=f"Weekly sales distribution by month (all years)")
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="",
    title={'x': 0.5}
    )
fig.show()

The boxplot of weekly sales by month displays a **seasonal pattern**, with a surge of sales in December (end-of-year Christmas holiday effect) and a drop in sales in January after the holidays.
During the rest of the year, certain months display higher variability (February, June, July, September, November), which could be explained by promotional campaigns or sales.  
We note an isolated outlier in January (maybe a data anomaly or an exceptional event).


In [38]:
# Boxplot of weekly sales by year x month
df['year_month'] = df['Year'].astype(str) + "-" + df['Month'].astype(str).str.zfill(2)

fig = px.box(df, 
                x='year_month', 
                y='Weekly_Sales', 
                width=1000, 
                height=500, 
                title=f"Weekly sales distribution by month and year")
fig.update_layout(
    xaxis_title="Year - Month",
    yaxis_title="",
    title={'x': 0.5}
    )
fig.show()


The boxplot of weekly sales by month and year displays as well a seasonal pattern, with peaks every December followed by notable drops in January. There is high variability in sales across months and years, suggesting the influence of either promotions or external factors.  

In [39]:
# Weekly sales statistics by holiday flag
df_sales_by_holiday = df.groupby('Holiday_Flag')['Weekly_Sales'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    min='min',
    q1=lambda x: x.quantile(0.25),
    q3=lambda x: x.quantile(0.75),
    max='max'
).reset_index()

df_sales_by_holiday['coef_variation'] = df_sales_by_holiday['std']/df_sales_by_holiday['mean']
print("Weekly sales statistics by holiday flag")
df_sales_by_holiday.round(2)


Weekly sales statistics by holiday flag


,Holiday_Flag,count,mean,median,std,min,q1,q3,max,coef_variation
0,0,105,1.25,1.41,0.68,0.27,0.53,1.85,2.77,0.54
1,1,8,1.44,1.70,0.63,0.56,0.90,1.87,2.08,0.44


In [40]:
# Boxplot of sales by holiday flag
fig = px.box(
    df,
    x='Holiday_Flag',
    y='Weekly_Sales',
    width=800,
    title="Distribution of weekly sales - holiday vs non-holiday weeks",
)

fig.update_layout(
    xaxis_title="Holiday flag (0 = non-holiday, 1 = holiday)",
    yaxis_title="Weekly sales (M USD)",
    title={'x': 0.5}
)

fig.show()


Holiday weeks tend to have higher and more consistent sales than non-holiday weeks:
- a median of 1.7M USD for holiday weeks vs 1.41 M USD for non-holiday weeks
- a lower coefficient of variation for holiday weeks (0.44) compared to non-holiday weeks (0.54).  
These observations support the inclusion of the `Holiday_Flag` in the model.

In [41]:
# Temporal relationship between weekly sales and exogeneous variables (fuel price, unemployment rate, consumer price index, temperature)

df_copy = df.copy().sort_values(['Date'])

fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[
        "Weekly sales",
        "Fuel price",
        "Unemployment rate",
        "Consumer Price Index (CPI)",
        "Temperature"
    ]
)

fig.add_trace(
    go.Scatter(
        x=df_copy['Date'],
        y=df_copy['Weekly_Sales'],
        mode='markers',
        name='Weekly sales'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_copy['Date'],
        y=df_copy['Fuel_Price'],
        mode='markers',
        name='Fuel price',
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_copy['Date'],
        y=df_copy['Unemployment'],
        mode='markers',
        name='Unemployment'
    ),
    row=3, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_copy['Date'],
        y=df_copy['CPI'],
        mode='markers',
        name='CPI'
    ),
    row=4, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_copy['Date'],
        y=df_copy['Temperature'],
        mode='markers',
        name='Temperature'
    ),
    row=5, col=1
)

fig.update_layout(
    height=900,
    width=800,
    title="Weekly sales and exogenous variables over time",
    title_x=0.5,
    showlegend=False
)

fig.update_xaxes(title_text="Date", row=5, col=1)
fig.show()


For exogeneous variables, we observe multiple values per week: the measurements may be reported at a local level.  
Fuel price and temperature show structured temporal patterns but there is no obvious parallel evolution (negative ou positive) with the weekly sales.  
The unemployement rate and CPI evolution over time does not show clear upward or downward trend, and again there is no obvious synchronization with weekly sales.


In [42]:
# Function which plots weekly sales and exogeneous variables for a single store

exog_vars = ['Fuel_Price', 'CPI', 'Temperature', 'Unemployment']

def plot_store_series(store_id, df):
    """
    Plot weekly sales and exogenous variables for a single store
    """
    # Select the store
    df_store = df.loc[df['Store'] == store_id].sort_values('Date').copy()
    
    # Number of subplots
    num_subplots = 1 + len(exog_vars)
    
    # Create subplots
    fig = make_subplots(
        rows=num_subplots,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=['Weekly sales'] + exog_vars
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_store['Date'],
            y=df_store['Weekly_Sales'],
            mode='lines+markers',
            name='Weekly sales',
            line=dict(dash='dot'),
            connectgaps=True
        ),
        row=1,
        col=1
    )
    
    for i, var in enumerate(exog_vars):
        fig.add_trace(
            go.Scatter(
                x=df_store['Date'],
                y=df_store[var],
                mode='lines+markers',
                name=var,
                line=dict(dash='dot'),
                connectgaps=True
            ),
            row=i+2,
            col=1
        )
    
    fig.update_layout(
        height=200*num_subplots,
        width=700,
        title_text=f'Weekly sales and exogenous variables - Store {store_id}',
        title={'x': 0.5},
    )
    
    fig.update_xaxes(title_text='Date', row=num_subplots, col=1)
    for i in range(1, num_subplots+1):
        fig.update_yaxes(title_text="", row=i, col=1)
    
    fig.show()


In [43]:
# Plot charts for several stores
for store in [1, 18, 19]:
    plot_store_series(str(store), df)

For a selection of stores, no clear pattern is observed between exogeneous variables and weekly stores.

However, even if no clear pattern is visible, the exogenous variables such as CPI, fuel price and unemployment can capture external economic factors that influence sales, through their impact on consumer behaviour or purchasing power. We will keep them in the model because of their economic rationale:
- CPI measures the overall level of consumer prices, thus is related to consumer purchasing power, and can influence consumer demand.
- Fuel prices affect business costs and consumer behaviour by its impact on transportation costs.
- Unemployment rate reflects labour market conditions and impacts incomes. A higher unemployment rate increases the risk of instable and lower income, which can lead to lower consumer spendings and lower retail sales.
- Temperature can influence consumer behaviour and demand for seasonal products (ex: during winter, a rise for heating products).

Based on this EDA and theoretical/economic expectations, we decide to include as explanatory variables: 
- all the exogeneous variables (fuel price, CPI, unemployment rate, temperature), 
- dummy variables for each store to account for store-specific effects
- the holiday flag and dummy variables for years and mmonths to account for seasonal and annual effects

## 4. Preprocessing with scikit-learn

### 4.1. Separate target variable Y from features X

In [44]:
features_to_include = ['Store', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Holiday_Flag', 'Year', 'Month']
Y = df[target]
X = df[features_to_include]

### 4.2. Train-test splitting
We use a standard random train-test split in a first approach (80% of the data for training and 20% for testing). However we are aware that this method is not strictly correct because it does not take into account the temporal order of observations and may lead to data leakage.

In [45]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [46]:
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of Y_train: {Y_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of Y_test: {Y_test.shape}")

Shape of X_train: (90, 8)
Shape of Y_train: (90,)
Shape of X_test: (23, 8)
Shape of Y_test: (23,)


### 4.3. Imputation of missing values, standardizing and one-hot encoding


In [47]:
# Define numeric and categorical features
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']  
categorical_features = ['Holiday_Flag', 'Store', 'Year', 'Month']

# Pipeline for numeric features

numeric_transformer = Pipeline(
    steps = [
        ("imputer", KNNImputer(n_neighbors=5)),  # missing values are imputed using the mean value from n_neighbors nearest neighbors
        ("scaler", StandardScaler())
    ]
)

# Pipeline for categorical features
categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),  # missing values will be replaced by most frequent value
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),  # first column will be dropped to avoid creating correlations between features. Unseen categories will be ignored
    ]
)

# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers = [
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 5. Baseline model: multivariate linear regression model

We estimate a multivariate linear regression model.

The performance of the model will be assessed using several metrics computed on both training and test sets:
- the coefficient of determination (R2): measures the variability in the target variable that is explained by the predictors;
- the mean absolute error (MAE): measures, on average, how far off the model's predictions deviatte from the actual values without being not overly influenced by a few large errors.
- the root mean squared error (RMSE) measure how far a model's predictions deviate from the actual values, with larger errors more heavily penalized than smaller ones.

Comparing training and test metrics will help to check for overfitting or underfitting issues.

In [48]:
# Pipeline for multivariate linear regression
pipeline_linreg = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])


In [49]:
# Apply the pipeline (preprocessings + model fitting)
pipeline_linreg.fit(X_train, Y_train)

# Predictions on train set and test set
Y_train_pred = pipeline_linreg.predict(X_train)
Y_test_pred  = pipeline_linreg.predict(X_test)


In [50]:
results = []

# Performance evaluation
R2_train = r2_score(Y_train, Y_train_pred)
R2_test = r2_score(Y_test, Y_test_pred)
RMSE_train = np.sqrt(mean_squared_error(Y_train, Y_train_pred))
RMSE_test = np.sqrt(mean_squared_error(Y_test, Y_test_pred))
MAE_train = mean_absolute_error(Y_train, Y_train_pred)
MAE_test = mean_absolute_error(Y_test, Y_test_pred)

metrics_linreg = {
    "Model": "Linear regression",
    "alpha": "na",
    "R2 - train": round(R2_train, 3),
    "R2 - test": round(R2_test, 3), 
    "RMSE - train": float(round(RMSE_train, 3)), 
    "RMSE - test": float(round(RMSE_test, 3)), 
    "MAE - train": round(MAE_train, 3), 
    "MAE - test": round(MAE_test, 3), 
}

results.append(metrics_linreg)

metrics_linreg

{'Model': 'Linear regression',
 'alpha': 'na',
 'R2 - train': 0.986,
 'R2 - test': 0.972,
 'RMSE - train': 0.08,
 'RMSE - test': 0.115,
 'MAE - train': 0.061,
 'MAE - test': 0.101}

In [51]:
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"Number of rows in train set : {X_train_transformed.shape[0]}")
print(f"Number of features in train set : {X_train_transformed.shape[1]}")


Number of rows in train set : 90
Number of features in train set : 36


The linear regression model achieves very high explanatory power: R² is very high both on training and test sets. The model explains 99% of the variance in the training set and 97% of the variance in the test set. R² test is slightly lower than R² train, but the gap is still acceptable. However the very good results on the training set (99%) is very suspicious.

Errors (RMSE and MAE) seem to be small compared to the scale of the weekly sales. On average the model's predictions on test set diverge from about 100k - 115 k USD, which represents about 8% to 9% of the mean weekly sales. 
RMSE (resp. MAE) increases by 44 % (resp. 66%) between training and test set, which may lead to think there is overfitting.

The number of features included in the model (and thus the model's complexity) is high compared to the number of observations available to train the model: 36 explanatory variables and the train set contains only 90 observations. It is not surprising to detect overfitting.
Also, because of the random train-test split used, these results may be too optimistic.

In [52]:
# Performance evaluation on linear model with cross-validation
scoring = ['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error']
cv = KFold(n_splits=5, shuffle=True, random_state=0)

results_cv = []
scores = cross_validate(pipeline_linreg, X_train, Y_train, cv=cv, scoring=scoring, return_train_score=False)

results_cv.append({
    "Model": "Linear regression - CV",
    "Mean R²": round(float(scores['test_r2'].mean()), 3),
    "Std R²": round(float(scores['test_r2'].std()), 3),
    "Mean RMSE": -round(float(scores['test_neg_root_mean_squared_error'].mean()), 3),
    "Std RMSE": round(float(scores['test_neg_root_mean_squared_error'].std()), 3),
    "Mean MAE": -round(float(scores['test_neg_mean_absolute_error'].mean()), 3),
    "Std MAE": round(float(scores['test_neg_mean_absolute_error'].std()))
})    

c:\Users\csil0\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning:

Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros

c:\Users\csil0\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning:

Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros



In [53]:
results_cv

[{'Model': 'Linear regression - CV',
  'Mean R²': 0.83,
  'Std R²': 0.109,
  'Mean RMSE': 0.242,
  'Std RMSE': 0.089,
  'Mean MAE': 0.149,
  'Std MAE': 0}]

The cross-validated R² score (83% with a standard deviation of 0.109) is lower than the R² on training and test sets on the train/test split we performed (R² train = 0.986 and R² test=0.972), suggesting overfitting. The very good results we obtained may be very specific to our train-test split.

In [54]:
# Scatterplot of predicted vs actual observations

# Create a DataFrame with true vs predicted
df_pred = pd.DataFrame({
    "Actual": Y_test,
    "Predicted": Y_test_pred
})

# Scatter plot
fig = px.scatter(
    df_pred,
    x="Actual",
    y="Predicted",
    height=500,
    width=500,
    title="Predicted vs actual weekly sales",
    labels={"Actual": "Actual sales (M USD)", "Predicted": "Predicted sales (M USD)"}
)

# Add diagonal line y=x for reference
fig.add_shape(
    type="line",
    x0=df_pred["Actual"].min(),
    y0=df_pred["Actual"].min(),
    x1=df_pred["Actual"].max(),
    y1=df_pred["Actual"].max(),
    line=dict(color="red", dash="dash")
)
fig.update_layout(
    title={'x': 0.5},
)  
fig.show()


The points are overall close to the diagonal line (which represents perfect prediction line y = x), meaning the model makes overall good predictions.

In [55]:
# Features importance

model = pipeline_linreg.named_steps['model']

coef_linreg = model.coef_

# Get feature names from the preprocessing pipeline
preprocessor = pipeline_linreg.named_steps['preprocessing']
numeric_feature_names = preprocessor.transformers_[0][1].get_feature_names_out()
categorical_feature_names = preprocessor.transformers_[1][1].get_feature_names_out()
feature_names = list(numeric_feature_names) + list(categorical_feature_names)

df_coef = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coef_linreg
})

df_coef['Absolute coefficient'] = df_coef['Coefficient'].abs()
df_coef = df_coef.sort_values('Absolute coefficient', ascending=False)

df_coef.round(2)

,Feature,Coefficient,Absolute coefficient
18,Store_5,-1.37,1.37
16,Store_3,-1.23,1.23
22,Store_9,-1.15,1.15
10,Store_16,-1.11,1.11
9,Store_15,-1.02,1.02
20,Store_7,-0.91,0.91
11,Store_17,-0.91,0.91
21,Store_8,-0.69,0.69
35,Month_12,0.63,0.63
8,Store_14,0.55,0.55


In [56]:
# Plot coefficients
fig = px.bar(
    df_coef,
    y='Coefficient',
    x='Feature',
    title='Feature importance - linear regression'
)
fig.update_layout(
    height=500,
    showlegend = False, 
    margin={'l': 120} # to avoid cropping of column names
)
fig.update_layout(
    title={'x': 0.5},
)  
fig.show()

We note that:
- The store-specific effects have a high impact on sales.  
- The month of December has a positive influence on sales.
- Exogeneous variables (CPI, unemployment rate, fuel price, temperature) have the lowest influence on sales.

# 6. Regularized regression models

To address overfitting, we use regularization to introduce a constraint on the model's coefficients.

## 6.1. Ridge model

In [57]:
# Pipeline for ridge regression with the default value (alpha=1)
pipeline_ridge = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=1.0))
])

# Apply the pipeline (preprocessings + model fitting)
pipeline_ridge.fit(X_train, Y_train)

# Predictions on train set and test set
Y_train_pred = pipeline_ridge.predict(X_train)
Y_test_pred  = pipeline_ridge.predict(X_test)


In [58]:
# Performance evaluation
R2_train = r2_score(Y_train, Y_train_pred)
R2_test = r2_score(Y_test, Y_test_pred)
RMSE_train = float(np.sqrt(mean_squared_error(Y_train, Y_train_pred)))
RMSE_test = float(np.sqrt(mean_squared_error(Y_test, Y_test_pred)))
MAE_train = mean_absolute_error(Y_train, Y_train_pred)
MAE_test = mean_absolute_error(Y_test, Y_test_pred)

metrics_ridge = {
    "Model": "Ridge regression",
    "alpha": 1,
    "R2 - train": round(R2_train, 3),
    "R2 - test": round(R2_test, 3), 
    "RMSE - train": round(RMSE_train, 3), 
    "RMSE - test": round(RMSE_test, 3), 
    "MAE - train": round(MAE_train, 3), 
    "MAE - test": round(MAE_test, 3), 
}

results.append(metrics_ridge)

metrics_ridge

{'Model': 'Ridge regression',
 'alpha': 1,
 'R2 - train': 0.943,
 'R2 - test': 0.933,
 'RMSE - train': 0.159,
 'RMSE - test': 0.178,
 'MAE - train': 0.128,
 'MAE - test': 0.136}

In [59]:
# Perform grid search to tune alpha

# Grid of values to be tested
param_grid_ridge = {
    'model__alpha': [i/100 for i in range(1, 110)] 
}

grid_ridge = GridSearchCV(
    pipeline_ridge,
    param_grid=param_grid_ridge,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)
grid_ridge.fit(X_train, Y_train)

# best_score_ is the mean cross-validation score for the best combination of hyperparameters tested during the grid search
# best_params_ is a dictionary containing the combination of hyperparameters that gave the best cross-validation score during the grid search

# best_estimator_ is the model trained on all training data using best_params_, the best combination of hyperparameters found during the grid search. 
# It's the version of the model to use for predictions, if this model is selected.

print("Best score CV (R²)", round(grid_ridge.best_score_, 3))
print("Best params", grid_ridge.best_params_)
print("Best estimator", grid_ridge.best_estimator_)

# Best model from GridSearchCV
best_model = grid_ridge.best_estimator_

# Predictions
Y_train_pred = best_model.predict(X_train)
Y_test_pred = best_model.predict(X_test)

# Performance metrics
R2_train = r2_score(Y_train, Y_train_pred)
R2_test = r2_score(Y_test, Y_test_pred)
RMSE_train = np.sqrt(mean_squared_error(Y_train, Y_train_pred))
RMSE_test = np.sqrt(mean_squared_error(Y_test, Y_test_pred))
MAE_train = mean_absolute_error(Y_train, Y_train_pred)
MAE_test = mean_absolute_error(Y_test, Y_test_pred)

metrics_ridge_gscv = {
    "Model": "Ridge GridsearchCV regression",
    "alpha": grid_ridge.best_params_['model__alpha'],
    "R2 - train": round(R2_train, 3),
    "R2 - test": round(R2_test, 3), 
    "RMSE - train": float(round(RMSE_train, 3)), 
    "RMSE - test": float(round(RMSE_test, 3)), 
    "MAE - train": round(MAE_train, 3), 
    "MAE - test": round(MAE_test, 3), 
}

results.append(metrics_ridge_gscv)

metrics_ridge_gscv

Best score CV (R²) 0.835
Best params {'model__alpha': 0.05}
Best estimator Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Temperature', 'Fuel_Price',
                                                   'CPI', 'Unemployment']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                               

{'Model': 'Ridge GridsearchCV regression',
 'alpha': 0.05,
 'R2 - train': 0.985,
 'R2 - test': 0.976,
 'RMSE - train': 0.082,
 'RMSE - test': 0.107,
 'MAE - train': 0.061,
 'MAE - test': 0.09}

In [60]:
# Features importance

model = best_model.named_steps['model']

coef_ridge = model.coef_

# Get feature names from the preprocessing pipeline
preprocessor = pipeline_linreg.named_steps['preprocessing']
numeric_feature_names = preprocessor.transformers_[0][1].get_feature_names_out()
categorical_feature_names = preprocessor.transformers_[1][1].get_feature_names_out()
feature_names = list(numeric_feature_names) + list(categorical_feature_names)

df_coef_ridge = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coef_ridge
})

df_coef_ridge['Absolute coefficient'] = df_coef_ridge['Coefficient'].abs()
df_coef_ridge = df_coef_ridge.sort_values('Absolute coefficient', ascending=False)

df_coef_ridge.round(2)

# Plot coefficients
fig = px.bar(
    df_coef_ridge,
    y='Coefficient',
    x='Feature',
    title='Feature importance - Ridge regression (alpha=0.05)'
)
fig.update_layout(
    height=500,
    showlegend = False, 
    margin={'l': 120} # to avoid cropping of column names
)
fig.update_layout(
    title={'x': 0.5},
)  
fig.show()

## 6.2. Lasso model

In [61]:
# Pipeline for lasso regression with the default value (alpha=1)
pipeline_lasso = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", Lasso(alpha=1.0))
])

# Apply the pipeline (preprocessings + model fitting)
pipeline_lasso.fit(X_train, Y_train)

# Predictions on train set and test set
Y_train_pred = pipeline_lasso.predict(X_train)
Y_test_pred  = pipeline_lasso.predict(X_test)


In [62]:
# Performance evaluation
R2_train = r2_score(Y_train, Y_train_pred)
R2_test = r2_score(Y_test, Y_test_pred)
RMSE_train = float(np.sqrt(mean_squared_error(Y_train, Y_train_pred)))
RMSE_test = float(np.sqrt(mean_squared_error(Y_test, Y_test_pred)))
MAE_train = mean_absolute_error(Y_train, Y_train_pred)
MAE_test = mean_absolute_error(Y_test, Y_test_pred)

metrics_lasso = {
    "Model": "Lasso regression",
    "alpha": 1,
    "R2 - train": round(R2_train, 3),
    "R2 - test": round(R2_test, 3), 
    "RMSE - train": round(RMSE_train, 3), 
    "RMSE - test": round(RMSE_test, 3), 
    "MAE - train": round(MAE_train, 3), 
    "MAE - test": round(MAE_test, 3), 
}

results.append(metrics_lasso)

metrics_lasso

{'Model': 'Lasso regression',
 'alpha': 1,
 'R2 - train': 0.0,
 'R2 - test': -0.02,
 'RMSE - train': 0.665,
 'RMSE - test': 0.697,
 'MAE - train': 0.591,
 'MAE - test': 0.644}

The Lasso regression model with the default hyperparameter (alpha=1) performs very poorly on both training and test sets. It even performs worse than a naive model which always predicts the mean weekly sales. The R² value on the training set is null, and is negative on the training set, indicating it has very poor predictive power. The RMSE and MAE are relatively high. The model seems to be too simple and underfit the data, maybe because the default hyperparameter leads to a strong regularization.

In [63]:
# Perform grid search to tune alpha

# Grid of values to be tested
param_grid_lasso = {
    'model__alpha': [0, 0.001, 0.005, 0.01, 0.1, 1, 5, 10, 20]
}

grid_lasso = GridSearchCV(
    pipeline_lasso,
    param_grid=param_grid_lasso,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)
grid_lasso.fit(X_train, Y_train)

,estimator,"Pipeline(step...l', Lasso())])"
,param_grid,"{'model__alpha': [0, 0.001, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [64]:
# best_score_ is the mean cross-validation score for the best combination of hyperparameters tested during the grid search
# best_params_ is a dictionary containing the combination of hyperparameters that gave the best cross-validation score during the grid search

# best_estimator_ is the model trained on all training data using best_params_, the best combination of hyperparameters found during the grid search. 
# It's the version of the model to use for predictions, if this model is selected.

print("Best score CV (R²)", round(grid_lasso.best_score_, 3))
print("Best params", grid_lasso.best_params_)
print("Best estimator", grid_lasso.best_estimator_)

Best score CV (R²) 0.843
Best params {'model__alpha': 0.001}
Best estimator Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Temperature', 'Fuel_Price',
                                                   'CPI', 'Unemployment']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                              

In [65]:
# Best model from GridSearchCV
best_model = grid_lasso.best_estimator_

# Predictions
Y_train_pred = best_model.predict(X_train)
Y_test_pred = best_model.predict(X_test)

# Performance metrics
R2_train = r2_score(Y_train, Y_train_pred)
R2_test = r2_score(Y_test, Y_test_pred)
RMSE_train = np.sqrt(mean_squared_error(Y_train, Y_train_pred))
RMSE_test = np.sqrt(mean_squared_error(Y_test, Y_test_pred))
MAE_train = mean_absolute_error(Y_train, Y_train_pred)
MAE_test = mean_absolute_error(Y_test, Y_test_pred)

metrics_lasso_gscv = {
    "Model": "Lasso GridsearchCV regression",
    "alpha": grid_lasso.best_params_['model__alpha'],
    "R2 - train": round(R2_train, 3),
    "R2 - test": round(R2_test, 3), 
    "RMSE - train": float(round(RMSE_train, 3)), 
    "RMSE - test": float(round(RMSE_test, 3)), 
    "MAE - train": round(MAE_train, 3), 
    "MAE - test": round(MAE_test, 3), 
}

results.append(metrics_lasso_gscv)

metrics_lasso_gscv

{'Model': 'Lasso GridsearchCV regression',
 'alpha': 0.001,
 'R2 - train': 0.983,
 'R2 - test': 0.98,
 'RMSE - train': 0.088,
 'RMSE - test': 0.097,
 'MAE - train': 0.065,
 'MAE - test': 0.079}

In [66]:
# Features importance

model = best_model.named_steps['model']

coef_lasso = model.coef_

# Get feature names from the preprocessing pipeline
preprocessor = pipeline_linreg.named_steps['preprocessing']
numeric_feature_names = preprocessor.transformers_[0][1].get_feature_names_out()
categorical_feature_names = preprocessor.transformers_[1][1].get_feature_names_out()
feature_names = list(numeric_feature_names) + list(categorical_feature_names)

df_coef_lasso = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coef_ridge
})

df_coef_lasso['Absolute coefficient'] = df_coef_lasso['Coefficient'].abs()
df_coef_lasso = df_coef_lasso.sort_values('Absolute coefficient', ascending=False)

df_coef_lasso.round(2)

# Plot coefficients
fig = px.bar(
    df_coef_lasso,
    y='Coefficient',
    x='Feature',
    title='Feature importance - Lasso regression (alpha=0.001)'
)
fig.update_layout(
    height=500,
    showlegend = False, 
    margin={'l': 120} # to avoid cropping of column names
)
fig.update_layout(
    title={'x': 0.5},
)  
fig.show()

## 6.3. Comparison

In [67]:
# Display results
df_results = pd.DataFrame(results)
df_results

,Model,alpha,R2 - train,R2 - test,RMSE - train,RMSE - test,MAE - train,MAE - test
0,Linear regression,na,0.986,0.972,0.080,0.115,0.061,0.101
1,Ridge regression,1,0.943,0.933,0.159,0.178,0.128,0.136
2,Ridge GridsearchCV regression,0.05,0.985,0.976,0.082,0.107,0.061,0.090
3,Lasso regression,1,0.000,-0.020,0.665,0.697,0.591,0.644
4,Lasso GridsearchCV regression,0.001,0.983,0.980,0.088,0.097,0.065,0.079


The regularized models (Ridge and Lasso) tuned with GridSearchCV seem to perform a bit better, with slightly less overfitting.
Using Gridsearch CV, the Lasso model has a very low regularization hyperparameter (alpha=0.001), meaning almost no regularization.

## 6. Limits and further analysis

* The analysis would be **more robust with a larger dataset**.

* The model suffers from **data leakage** because of improper data splitting, which may lead to overestimate the model performance.

    In panel datasets, the hypothesis of independent and identically distributed observations (iid) over both the time and cross-sectional dimensions is violated. Here two types of leakage can occur: 
    - a temporal leakage: the random split can lead to include data from the oldest dates in the test set, and data from latest dates in the train set, so the model sees both old and recent information during the training process;
    - a cross-sectional leakage: the same store can appear in both the train and test sets. But it is less problematic if the goal is to predict sales for an existing store.

    Consequently, the model will not probably encounter unseen data during the prediction step. Worse, the model may end up predicting past values of the target using information from latest dates, which doesn't make any sense.

    To address the temporal leakage, we could split non-randomly at the time level: the earliest time periods would appear in the train set, the latest periods would appear in the test set.